In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm import OpenAILLM, OllamaLLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.generation.prompts import ERExtractionTemplate
from dotenv import load_dotenv
import pandas as pd
from langchain_core.documents import Document
from langchain_community.llms import Ollama
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever, Text2CypherRetriever
from neo4j_graphrag.schema import get_schema
from neo4j_graphrag.generation import GraphRAG


import os, time, asyncio, glob, csv

## 1. Initialize Graph Database 

In [2]:
load_dotenv(dotenv_path="../.env")

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE_NAME = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

### 1.1 Call BGE-M3 Embedding model

In [3]:
import torch
from FlagEmbedding import BGEM3FlagModel

# เช็คว่ามี GPU (CUDA) หรือไม่
# ถ้ามีให้ใช้ 'cuda' ถ้าไม่มีให้ใช้ 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_fp16 = True if device == 'cuda' else False  # fp16 ส่วนใหญ่รองรับเฉพาะบน GPU

print(f"กำลังรันโมเดลบน: {device.upper()}")

# 2. Initialize Model
embedder = BGEM3FlagModel(
    'VISAI-AI/nitibench-ccl-human-finetuned-bge-m3', 
    use_fp16=use_fp16, 
    device=device 
)

กำลังรันโมเดลบน: CUDA


Fetching 23 files: 100%|██████████| 23/23 [00:00<00:00, 22985.23it/s]


### 1.2 Wrap embedding model
เนื่องจาก Embedding model ที่จะใช้ใน Neo4J (เรียกจาก library Neo4J โดยตรง ไม่ใช่ langchain) ไม่สามารถเรียกใช้ Embedding model BGE-M3 ได้โดยตรง ตัว Library รองรับแค่พวก Commercial Models e.g. openai จึงต้องมีการ Wrap เพื่อให้ใช้งานได้ 

In [4]:
from neo4j_graphrag.embeddings import Embedder

class Neo4jCompatibleEmbedder(Embedder):
    def __init__(self, original_embedder):
        self.original_embedder = original_embedder

    def embed_query(self, text: str) -> list[float]:
        # 1. เรียกโมเดล M3 (สมมติว่าใช้ .encode)
        result = self.original_embedder.encode(text)
        
        # 2. ตรวจสอบว่าผลลัพธ์เป็น dict หรือไม่ (BGE-M3 มักคืนค่าแบบนี้ถ้าไม่ได้ตั้ง return_dense_only=True)
        if isinstance(result, dict):
            # ดึงเฉพาะ Dense Vector ออกมา
            embedding = result.get('dense_vecs')
        else:
            embedding = result

        # 3. แปลงจาก numpy array เป็น list เพื่อให้ Pydantic/Neo4j ยอมรับ
        if hasattr(embedding, "tolist"):
            return embedding.tolist()
        
        # กรณีเป็น list อยู่แล้วแต่สมาชิกข้างในอาจเป็น numpy types ให้วนลูปแปลง (ถ้าจำเป็น)
        return [float(x) for x in embedding]

    def embed_nodes(self, nodes):
        for node in nodes:
            # node.text คือเนื้อหาที่ถูก Chunk แล้ว
            node.embedding = self.embed_query(node.text)

# การใช้งาน
# ระวังอย่ารันบรรทัดนี้ซ้ำ (เพราะจะกลายเป็น wrap ซ้อน wrap)
neo4j_embedder = Neo4jCompatibleEmbedder(embedder)

In [5]:
df = pd.read_parquet('../data/processed/nitibench_cleaned_2026-03-17.parquet')
df.head()

,law_name,section_content,section_num
0,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มา...,132
1,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู...,1598/5
2,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876\nถ้าผู้รั...,876
3,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1030\nถ้าผู้เ...,1030
4,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 849\nการรับเง...,849


## 2. Chunking to documents

In [6]:
def batch_document_append(df,start=0, end=len(df)):
    documents = []
    for index, row in df.iloc[start:end,:].iterrows():
        # สร้างเนื้อหาที่มีบริบทชัดเจนในตัวเอง
        content = f"กฎหมาย: {row['law_name']}\nมาตรา: {row['section_num']}\nเนื้อหา: {row['section_content']}"
        
        # เก็บ Metadata เพื่อใช้ในภายหลัง
        metadata = {
            "law_name": row['law_name'],
            "section_num": str(row['section_num']),
        }
        documents.append(Document(page_content=content, metadata=metadata))
    return documents

documents = batch_document_append(df, start=0, end=50)
documents

[Document(metadata={'law_name': 'พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', 'section_num': '132'}, page_content='กฎหมาย: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546\nมาตรา: 132\nเนื้อหา: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'),
 Document(metadata={'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 'section_num': '1598/5'}, page_content='กฎหมาย: ประมวลกฎหมายแพ่งและพาณิชย์\nมาตรา: 1598/5\nเนื้อหา: ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู้อยู่ในปกครองรู้จักผิดชอบและมีอายุไม่ต่ำกว่าสิบห้าปีบริบูรณ์เมื่อผู้ปกครองจะทำกิจการใดที่สำคัญ ให้ปรึกษาหารือผู้อยู่ในปกครองก่อนเท่าที่จะทำได้\nการที่ผู้อยู่ในปกครองได้ยินยอมด้วยนั้นหาคุ้มผู้ปกครองให้พ้นจากความรับผิดไม่'),
 Document(metadata={'law_name': 'ประมวลกฎหมายแพ่งและพาณิชย์', 'section_num': '87

## 3. Select LLM

In [34]:
from neo4j_graphrag.llm import OllamaLLM
# llm = OllamaLLM(
#     model_name="llama3",
#     # model_params={"options": {"temperature": 0}, "format": "json"},
#     host="http://localhost:11434",  # when using a remote server
# )

os.environ["OPENAI_API_KEY"] = os.getenv("openrouter_API_key") # ใส่ OpenRouter Key ของคุณตรงนี้

# 2. เรียกใช้งาน โดยใส่ base_url กำกับไว้ข้างนอก
llm = OpenAILLM(
    model_name="openai/gpt-4o-mini", 
    base_url="https://openrouter.ai/api/v1",
    model_params={
        "temperature": 0
    }
)
llm.invoke("say hello-world")

LLMResponse(content='Hello, World!')

## 4. Graph Construction

In [8]:
# llm = ChatOpenAI(api_key=os.getenv("openrouter_API_key") ,temperature=0, model="openai/gpt-4o-mini", base_url="https://openrouter.ai/api/v1")
# llm = OllamaLLM(model="llama3", )
entities = [
    {"label": "law_name", "properties": [{"name": "text", "type": "STRING"}]},
    {"label": "section_num", "properties": [{"name": "text", "type": "STRING"}]},
]

relations = [
    {"label": "MENTIONS", "source": "law_name", "target": "section_num"},
    {"label": "REFERENCES", "source": "section_num", "target": "section_num"},
]


kg_builder = SimpleKGPipeline(
    driver=driver,
    llm=llm,
    embedder=neo4j_embedder,
    entities=entities,
    relations=relations,
    neo4j_database=DATABASE_NAME,
    from_pdf=False
    )

In [ ]:
for i in range(10):
    await kg_builder.run_async(text=documents[i].page_content, document_metadata=documents[i].metadata,)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


## 5. Retrieval

### 5.1 Vector Retriever เหมือน RAG ธรรมดา

In [14]:
# Initialize the retriever
retriever = VectorRetriever(
     driver,
     index_name= "text_embeddings",
     embedder=neo4j_embedder,
     return_properties=["text"]
)
query = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
result = retriever.search(query_text=query, top_k=10)

In [25]:
# ดึงเนื้อหาของอันแรก
first_result = result.items[0].content
# ดึงค่า Score (ความเหมือน)
first_score = result.items[0].metadata['score']

print(f"Score: {first_score}")
print(f"Content: {first_result}")

Score: 0.9035258293151855
Content: {'text': 'กฎหมาย: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546\nมาตรา: 132\nเนื้อหา: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'}


### 5.2 Vector Cypher Retriever 
- ใช้ Cosine similarity เทียบความคล้าย
- Cypher query เข้าไป (ต้องมีการกำหนดเอง ดีที่สุดคืออิงจาก Schema) 

In [27]:
schema = get_schema(driver)
schema

'Node properties:\nDocument {section_num: STRING, document_type: STRING, path: STRING, createdAt: STRING, law_name: STRING}\nChunk {index: INTEGER, text: STRING, embedding: LIST}\nlaw_name {text: STRING}\nsection_num {text: STRING}\nRelationship properties:\n\nThe relationships:\n(:Chunk)-[:FROM_DOCUMENT]->(:Document)\n(:law_name)-[:FROM_CHUNK]->(:Chunk)\n(:law_name)-[:MENTIONS]->(:section_num)\n(:law_name)-[:REFERENCES]->(:section_num)\n(:section_num)-[:FROM_CHUNK]->(:Chunk)'

In [ ]:
chunk_to_law_query = """
MATCH (node:Chunk)
OPTIONAL MATCH (law:law_name)-[:FROM_CHUNK]->(node)
OPTIONAL MATCH (law)-[:MENTIONS]->(section:section_num)
RETURN 
    node.text AS text,
    law.text AS law_title,
    collect(section.text) AS related_sections,
    node.index AS chunk_index
"""

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name="text_embeddings",
    embedder=neo4j_embedder,
    retrieval_query=chunk_to_law_query
)

result = vector_cypher_retriever.search(query_text=query, top_k=10)
for item in result.items:
    print(item.content)

Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \nMATCH (node:Chunk)\nOPTIONAL MATCH (law:law_name)-[:FROM_CHUNK]->(node)\nOPTIONAL MATCH (law)-[:MENTIONS]->(section:section_num)\nRETURN \n    node.text AS text,\n    law.text AS law_title,\n    collect(section.text) AS related_sections,\n    node.index AS chunk_index\n'


<Record text='กฎหมาย: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546\nมาตรา: 132\nเนื้อหา: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน' law_title='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546' related_sections=['132'] chunk_index=0>
<Record text='กฎหมาย: ประมวลกฎหมายแพ่งและพาณิชย์\nมาตรา: 1598/5\nเนื้อหา: ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู้อยู่ในปกครองรู้จักผิดชอบและมีอายุไม่ต่ำกว่าสิบห้าปีบริบูรณ์เมื่อผู้ปกครองจะทำกิจการใดที่สำคัญ ให้ปรึกษาหารือผู้อยู่ในปกครองก่อนเท่าที่จะทำได้\nการที่ผู้อยู่ในปกครองได้ยินยอมด้วยนั้นหาคุ้มผู้ปกครองให้พ้นจากความรับผิดไม่' law_title='ประมวลกฎหมายแพ่งและพาณิชย์' related_sections=['1598/5'] chunk_index=0>
<Record text='กฎหมาย: ประมวลกฎหมายแพ่งและพาณิชย์\nมาตรา: 1030\nเนื้อหา: ประมวลกฎหมายแพ่งและพาณิช

In [38]:
result = GraphRAG(llm=llm,retriever=vector_cypher_retriever)
print(result.search(query_text=query).answer)

Received notification from DBMS server: <GqlStatusObject gql_status='01G11', status_description='warn: null value eliminated in set function', position=None, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k \nMATCH (node:Chunk)\nOPTIONAL MATCH (law:law_name)-[:FROM_CHUNK]->(node)\nOPTIONAL MATCH (law)-[:MENTIONS]->(section:section_num)\nRETURN \n    node.text AS text,\n    law.text AS law_title,\n    collect(section.text) AS related_sections,\n    node.index AS chunk_index\n'


ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาต จะต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน ตามพระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132.
